In [2]:
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

class MNISTMock:
    def __init__(self):
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        
        self.train_images = x_train.reshape(-1, 784).astype(np.float32) / 255.0
        self.test_images = x_test.reshape(-1, 784).astype(np.float32) / 255.0
        
        self.train_labels = np.eye(10)[y_train].astype(np.float32)
        self.test_labels = np.eye(10)[y_test].astype(np.float32)
        
        self._index_in_epoch = 0
        self._num_examples = self.train_images.shape[0]

        class Dataset: pass
        self.train = Dataset()
        self.train.images, self.train.labels = self.train_images, self.train_labels
        self.train.next_batch = self.next_batch
        
        self.test = Dataset()
        self.test.images, self.test.labels = self.test_images, self.test_labels

    def next_batch(self, batch_size):
        start = self._index_in_epoch
        self._index_in_epoch += batch_size
        if self._index_in_epoch > self._num_examples:
            # 跑完一轮后打乱数据
            perm = np.arange(self._num_examples)
            np.random.shuffle(perm)
            self.train_images = self.train_images[perm]
            self.train_labels = self.train_labels[perm]
            start = 0
            self._index_in_epoch = batch_size
        end = self._index_in_epoch
        return self.train_images[start:end], self.train_labels[start:end]

mnist = MNISTMock()

learning_rate = 1e-4
keep_prob_rate = 0.7 
max_epoch = 2000
def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf.equal(tf.argmax(y_pre,1), tf.argmax(v_ys,1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
    return result

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # strides=[1, y方向步长, x方向步长, 1]
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    # ksize=[1, 池化高度, 池化宽度, 1], strides=[1, y方向步长, x方向步长, 1]
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# define placeholder for inputs to network
xs = tf.placeholder(tf.float32, [None, 784])/255.
ys = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 卷积层 1
W_conv1 = weight_variable([7, 7, 1, 32])                      
b_conv1 = bias_variable([32])                                 
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)  #卷积后接入ReLU激活函数
h_pool1 = max_pool_2x2(h_conv1)                           # 经过2x2最大池化               

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])                        
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)  # 卷积后接入ReLU激活函数
h_pool2 = max_pool_2x2(h_conv2)                           # 经过2x2最大池化

#  全连接层 1
## fc1 layer ##
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
## fc2 layer ##
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)


# 交叉熵函数
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction),
                                              reduction_indices=[1]))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    init = tf.global_variables_initializer()
    sess.run(init)
    
    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob:keep_prob_rate})
        if i % 100 == 0:
            print(compute_accuracy(
                mnist.test.images[:1000], mnist.test.labels[:1000]))


Instructions for updating:
non-resource variables are not supported in the long term
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
0.1
0.86
0.903
0.93
0.943
0.949
0.947
0.96
0.966
0.969
0.975
0.975
0.973
0.978
0.982
0.98
0.979
0.983
0.979
0.982
